
Tujuan notebook ini: **menghilangkan error incompatible** yang sering muncul di Vast.ai (torch/torchvision mismatch, numpy/opencv/albumentations conflict).

Prinsip:
- **Tidak pakai Albumentations** (sering jadi sumber conflict).
- Augmentasi dibuat dengan PIL + torchvision functional (mask‑safe).
- Inference & submission pakai OpenCV **headless** (kalau belum ada, install sekali).


In [11]:
import sys, platform
print("python:", sys.version)
print("platform:", platform.platform())

import numpy as np
import torch
print("numpy:", np.__version__)
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


python: 3.12.12 | packaged by conda-forge | (main, Oct 22 2025, 23:25:55) [GCC 14.3.0]
platform: Linux-6.5.0-45-generic-x86_64-with-glibc2.39
numpy: 2.4.1
torch: 2.10.0+cu128
cuda: True
gpu: NVIDIA GeForce RTX 2060 SUPER


In [12]:
import torchvision
print("torchvision:", torchvision.__version__)


torchvision: 0.25.0+cu128


In [13]:
import os, re, random, math, glob
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from PIL import Image
import torchvision.transforms.functional as TF


In [14]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 42
seed_everything(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)


DEVICE: cuda


In [15]:

train_img_dir = DATA_ROOT / "train" / "images"
train_msk_dir = DATA_ROOT / "train" / "mask"
test_img_dir  = DATA_ROOT / "test"  / "images"

assert train_img_dir.exists(), f"train_img_dir not found: {train_img_dir}"
assert train_msk_dir.exists(), f"train_msk_dir not found: {train_msk_dir}"
assert test_img_dir.exists(),  f"test_img_dir not found: {test_img_dir}"

print("train images:", len(list(train_img_dir.glob("*"))))
print("train masks :", len(list(train_msk_dir.glob("*"))))
print("test images :", len(list(test_img_dir.glob("*"))))


train images: 498
train masks : 498
test images : 295


In [16]:
def find_mask_for_image(img_path: str):
    base = os.path.splitext(os.path.basename(img_path))[0]  # train_157
    m = re.search(r"(\d+)$", base)
    if m is None:
        return None
    idx = m.group(1)
    for ext in ("png", "jpg", "jpeg"):
        c = train_msk_dir / f"mask_{idx}.{ext}"
        if c.exists():
            return str(c)
    return None

def encode_rle(mask: np.ndarray, pos_value: int = 255) -> str:
    binary = (mask == pos_value).astype(np.uint8)
    pixels = binary.T.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[0::2]
    return " ".join(str(x) for x in runs)

def dice_coef_np(pred_bool: np.ndarray, gt_bool: np.ndarray, eps=1e-6) -> float:
    inter = np.logical_and(pred_bool, gt_bool).sum()
    s = pred_bool.sum() + gt_bool.sum()
    if s == 0:
        return 1.0
    return float((2 * inter + eps) / (s + eps))


In [17]:
VAL_RATIO = 0.2

all_imgs = sorted([p.as_posix() for p in train_img_dir.glob("*")])
assert len(all_imgs) > 0, "No training images found."

random.Random(SEED).shuffle(all_imgs)
n_val = int(len(all_imgs) * VAL_RATIO)
val_imgs = all_imgs[:n_val]
trn_imgs = all_imgs[n_val:]

print("train:", len(trn_imgs), "val:", len(val_imgs))


train: 399 val: 99


In [18]:
import numpy as np
import cv2
import torch

# ImageNet normalize constants
IMNET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMNET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

@torch.no_grad()
def predict_prob(model, img_bgr, img_size=640, device=None):
    """
    Input: img_bgr (H,W,3) uint8 BGR (from cv2)
    Output: prob map (img_size, img_size) float32 0..1
    """
    if device is None:
        device = next(model.parameters()).device

    # BGR -> RGB
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # resize to square model input
    img_rs = cv2.resize(img_rgb, (img_size, img_size), interpolation=cv2.INTER_LINEAR).astype(np.float32) / 255.0

    # normalize
    img_rs = (img_rs - IMNET_MEAN) / IMNET_STD

    # HWC -> CHW, add batch
    x = torch.from_numpy(img_rs.transpose(2,0,1)).unsqueeze(0).to(device, non_blocking=True).float()

    # forward -> sigmoid
    logits = model(x)               # (1,1,S,S)
    prob = torch.sigmoid(logits)[0,0].detach().cpu().numpy().astype(np.float32)  # (S,S)

    return prob


In [19]:
@torch.no_grad()
def predict_prob_original(model, img_bgr, img_size=640, device=None):
    """
    Predict prob in model size (S,S) then upscale to original (H0,W0).
    """
    H0, W0 = img_bgr.shape[:2]
    prob_s = predict_prob(model, img_bgr, img_size=img_size, device=device)  # (S,S)
    prob_up = cv2.resize(prob_s, (W0, H0), interpolation=cv2.INTER_LINEAR)
    return prob_up  # (H0,W0)

@torch.no_grad()
def predict_prob_original_hflip(model, img_bgr, img_size=640, device=None):
    img_flip = cv2.flip(img_bgr, 1)
    prob_flip = predict_prob_original(model, img_flip, img_size=img_size, device=device)
    prob_unflip = cv2.flip(prob_flip, 1)
    return prob_unflip

@torch.no_grad()
def predict_prob_mean_multiscale_tta(model, img_bgr, sizes=(640,768), use_hflip=True, device=None):
    probs = []
    for s in sizes:
        probs.append(predict_prob_original(model, img_bgr, img_size=s, device=device))
        if use_hflip:
            probs.append(predict_prob_original_hflip(model, img_bgr, img_size=s, device=device))
    prob_mean = np.mean(np.stack(probs, axis=0), axis=0)  # (H0,W0)
    return prob_mean


In [ ]:
import numpy as np

def predict_prob_combo(model, img_bgr, sizes=(640,768), use_hflip=True, weights=None, device=None):
    """
    Combine multi-scale + optional hflip into ONE prob map (H0,W0).
    Ordering of probs:
      for each size s in sizes:
        prob(s), then (if use_hflip) prob(s,hflip)
    weights optional: list length must match K probs.
    """
    probs = []
    for s in sizes:
        probs.append(predict_prob_original(model, img_bgr, img_size=int(s), device=device))
        if use_hflip:
            probs.append(predict_prob_original_hflip(model, img_bgr, img_size=int(s), device=device))

    stack = np.stack(probs, axis=0).astype(np.float32)  # (K,H0,W0)

    if weights is None:
        return stack.mean(axis=0)

    w = np.array(weights, dtype=np.float32)
    assert len(w) == stack.shape[0], f"weights length {len(w)} != K {stack.shape[0]}"
    w = w / (w.sum() + 1e-8)
    return (stack * w[:, None, None]).sum(axis=0)


In [ ]:
from tqdm import tqdm

def dice_bool(pred_bool: np.ndarray, gt_bool: np.ndarray, eps=1e-6) -> float:
    inter = np.logical_and(pred_bool, gt_bool).sum()
    s = pred_bool.sum() + gt_bool.sum()
    if s == 0:
        return 1.0
    return float((2.0 * inter + eps) / (s + eps))

@torch.no_grad()
def eval_val_dice_tta(model, val_img_paths, thr, sizes=(640,768), use_hflip=True, device=None):
    model.eval()
    dices = []
    empty = 0
    total = 0

    for img_path in tqdm(val_img_paths, desc=f"VAL TTA thr={thr:.3f}", leave=False):
        img_bgr = cv2.imread(img_path)
        if img_bgr is None:
            continue

        # GT original
        mpath = find_mask_for_image(img_path)
        gt = cv2.imread(mpath, cv2.IMREAD_GRAYSCALE)
        if gt is None:
            continue
        gt_bool = (gt == 255)

        prob_mean = predict_prob_mean_multiscale_tta(
            model, img_bgr, sizes=sizes, use_hflip=use_hflip, device=device
        )
        pred_bool = (prob_mean >= float(thr))
        if pred_bool.sum() == 0:
            empty += 1

        dices.append(dice_bool(pred_bool, gt_bool))
        total += 1

    mean_dice = float(np.mean(dices)) if dices else 0.0
    empty_rate = float(empty / max(1, total))
    return mean_dice, empty_rate



In [21]:
IMG_SIZE = 640

IMNET_MEAN = (0.485, 0.456, 0.406)
IMNET_STD  = (0.229, 0.224, 0.225)

@dataclass
class AugConfig:
    hflip_p: float = 0.5
    vflip_p: float = 0.15
    rot_deg: float = 10.0
    translate: float = 0.03  # fraction
    scale_min: float = 0.95
    scale_max: float = 1.05
    brightness: float = 0.15
    contrast: float = 0.15
    saturation: float = 0.05
    hue: float = 0.02

AUG = AugConfig()

def pil_to_tensor_imnet(img_pil: Image.Image) -> torch.Tensor:
    x = TF.to_tensor(img_pil)  # [0,1] float32 CHW
    x = TF.normalize(x, IMNET_MEAN, IMNET_STD)
    return x

def mask_pil_to_tensor(mask_pil: Image.Image) -> torch.Tensor:
    m = np.array(mask_pil, dtype=np.uint8)
    m = (m == 255).astype(np.float32)  # 0/1
    return torch.from_numpy(m).unsqueeze(0)  # (1,H,W)

def apply_train_aug(img: Image.Image, mask: Image.Image, cfg: AugConfig):
    # resize first
    img = img.resize((IMG_SIZE, IMG_SIZE), resample=Image.BILINEAR)
    mask = mask.resize((IMG_SIZE, IMG_SIZE), resample=Image.NEAREST)

    # flips
    if random.random() < cfg.hflip_p:
        img = TF.hflip(img); mask = TF.hflip(mask)
    if random.random() < cfg.vflip_p:
        img = TF.vflip(img); mask = TF.vflip(mask)

    # affine (mask-safe)
    angle = random.uniform(-cfg.rot_deg, cfg.rot_deg)
    max_tx = int(cfg.translate * IMG_SIZE)
    max_ty = int(cfg.translate * IMG_SIZE)
    trans = (random.randint(-max_tx, max_tx), random.randint(-max_ty, max_ty))
    scale = random.uniform(cfg.scale_min, cfg.scale_max)
    shear = 0.0

    img = TF.affine(img, angle=angle, translate=trans, scale=scale, shear=shear,
                    interpolation=TF.InterpolationMode.BILINEAR)
    mask = TF.affine(mask, angle=angle, translate=trans, scale=scale, shear=shear,
                     interpolation=TF.InterpolationMode.NEAREST)

    # color jitter (image only)
    img = TF.adjust_brightness(img, 1.0 + random.uniform(-cfg.brightness, cfg.brightness))
    img = TF.adjust_contrast(img,   1.0 + random.uniform(-cfg.contrast, cfg.contrast))
    if cfg.saturation > 0:
        img = TF.adjust_saturation(img, 1.0 + random.uniform(-cfg.saturation, cfg.saturation))
    if cfg.hue > 0:
        img = TF.adjust_hue(img, random.uniform(-cfg.hue, cfg.hue))

    return img, mask

def apply_val_tf(img: Image.Image, mask: Image.Image):
    img = img.resize((IMG_SIZE, IMG_SIZE), resample=Image.BILINEAR)
    mask = mask.resize((IMG_SIZE, IMG_SIZE), resample=Image.NEAREST)
    return img, mask


In [22]:
class PotholeDataset(Dataset):
    def __init__(self, img_paths, is_train: bool):
        self.img_paths = img_paths
        self.is_train = is_train

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        mask_path = find_mask_for_image(img_path)
        if mask_path is None:
            raise FileNotFoundError(f"Mask not found for {img_path}")

        img = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        if self.is_train:
            img, mask = apply_train_aug(img, mask, AUG)
        else:
            img, mask = apply_val_tf(img, mask)

        x = pil_to_tensor_imnet(img)
        y = mask_pil_to_tensor(mask)
        return x, y, img_path

BATCH = 8
NUM_WORKERS = 4

trn_ds = PotholeDataset(trn_imgs, is_train=True)
val_ds = PotholeDataset(val_imgs, is_train=False)

trn_loader = DataLoader(trn_ds, batch_size=BATCH, shuffle=True, num_workers=NUM_WORKERS,
                        pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS,
                        pin_memory=True)

print("train batches:", len(trn_loader), "val batches:", len(val_loader))


train batches: 49 val batches: 13


## Model: ResNet34‑UNet (pretrained)


In [23]:
from torchvision.models import resnet34, ResNet34_Weights

class ConvBNReLU(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, p=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, k, padding=p, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.conv1 = ConvBNReLU(in_ch + skip_ch, out_ch)
        self.conv2 = ConvBNReLU(out_ch, out_ch)
    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        x = self.conv1(x)
        x = self.conv2(x)
        return x

class ResNet34_UNet(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        weights = ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
        m = resnet34(weights=weights)

        self.inp = nn.Sequential(m.conv1, m.bn1, m.relu)  # 64, H/2
        self.pool = m.maxpool                              # H/4

        self.enc1 = m.layer1  # 64
        self.enc2 = m.layer2  # 128
        self.enc3 = m.layer3  # 256
        self.enc4 = m.layer4  # 512

        self.center = ConvBNReLU(512, 512)
        self.up4 = UpBlock(512, 256, 256)
        self.up3 = UpBlock(256, 128, 128)
        self.up2 = UpBlock(128,  64,  64)
        self.up1 = UpBlock( 64,  64,  32)

        self.head = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, x):
        s1 = self.inp(x)     # 64, H/2
        x = self.pool(s1)    # H/4
        s2 = self.enc1(x)    # 64, H/4
        s3 = self.enc2(s2)   # 128, H/8
        s4 = self.enc3(s3)   # 256, H/16
        s5 = self.enc4(s4)   # 512, H/32

        c  = self.center(s5)
        d4 = self.up4(c,  s4)
        d3 = self.up3(d4, s3)
        d2 = self.up2(d3, s2)
        d1 = self.up1(d2, s1)

        logits = self.head(d1)
        logits = F.interpolate(logits, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)
        return logits

model = ResNet34_UNet(pretrained=True).to(DEVICE)
print("model ready")


model ready


In [24]:
# ====== loss + eval ======
class DiceLoss(nn.Module):
    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps = eps
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits).view(logits.size(0), -1)
        targets = targets.view(targets.size(0), -1)
        inter = (probs * targets).sum(dim=1)
        union = probs.sum(dim=1) + targets.sum(dim=1)
        dice = (2*inter + self.eps) / (union + self.eps)
        return 1 - dice.mean()

bce = nn.BCEWithLogitsLoss()

dice_loss = DiceLoss()
def mixed_loss(logits, targets, bce_w=0.5, dice_w=0.5):
    return bce_w * bce(logits, targets) + dice_w * dice_loss(logits, targets)


@torch.no_grad()
def eval_val_dice_multi(model, loader, thrs=(0.60, 0.65, 0.70)):
    model.eval()
    dices = {float(t): [] for t in thrs}
    empties = {float(t): 0 for t in thrs}
    total = 0

    for x, y, _ in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        prob = torch.sigmoid(model(x)).detach().cpu().numpy()
        gt   = y.detach().cpu().numpy()

        B = prob.shape[0]
        total += B
        for i in range(B):
            p = prob[i,0]
            g = (gt[i,0] >= 0.5)
            for t in thrs:
                pred = (p >= float(t))
                if pred.sum() == 0:
                    empties[float(t)] += 1
                dices[float(t)].append(dice_coef_np(pred, g))

    mean_dice = {t: float(np.mean(dices[t])) for t in dices}
    empty_rate = {t: float(empties[t] / max(1, total)) for t in empties}
    return mean_dice, empty_rate


## Training (freeze→unfreeze, 2 LR groups, cosine, early stopping)


In [25]:
import numpy as np

EPOCHS = 50
PATIENCE = 12
LR_DEC = 1e-3
LR_ENC = 1e-4
WARMUP_EPOCHS = 3
# Freeze encoder dulu
set_encoder_trainable(model, False)

# Encoder params (explicit)
enc_params = (
    list(model.inp.parameters()) +
    list(model.enc1.parameters()) +
    list(model.enc2.parameters()) +
    list(model.enc3.parameters()) +
    list(model.enc4.parameters())
)

# === FIX: filter by object identity, not tensor equality ===
enc_param_ids = set(map(id, enc_params))
dec_params = [p for p in model.parameters() if id(p) not in enc_param_ids]

# (opsional) sanity check: pastikan tidak overlap & tidak kosong
print("enc params:", sum(p.numel() for p in enc_params))
print("dec params:", sum(p.numel() for p in dec_params))
assert len(enc_params) > 0 and len(dec_params) > 0
assert enc_param_ids.isdisjoint(set(map(id, dec_params)))

opt = torch.optim.AdamW(
    [
        {"params": enc_params, "lr": LR_ENC},
        {"params": dec_params, "lr": LR_DEC},
    ],
    weight_decay=1e-4
)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

best_path = "./resunet_best.pt"
best_val = -1.0
no_improve = 0
THRS_MON = (0.60, 0.65, 0.70)

print("Start training...")
for epoch in range(1, EPOCHS+1):
    model.train()
    if epoch == WARMUP_EPOCHS + 1:
        set_encoder_trainable(model, True)
        print("Unfreeze encoder (encoder LR tetap kecil via 2LR groups).")

    losses = []
    for x, y, _ in trn_loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        opt.zero_grad(set_to_none=True)
        logits = model(x)
        loss = mixed_loss(logits, y, bce_w=0.5, dice_w=0.5)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        losses.append(float(loss.item()))

    sched.step()
    tr_loss = float(np.mean(losses)) if losses else 0.0

    mean_dice, empty_rate = eval_val_dice_multi(model, val_loader, thrs=THRS_MON)
    val_score = mean_dice[0.65]

    lr_enc = opt.param_groups[0]["lr"]
    lr_dec = opt.param_groups[1]["lr"]

    print(
        f"Epoch {epoch:03d} | train_loss={tr_loss:.4f} | "
        f"val@0.60={mean_dice[0.60]:.4f} (empty {empty_rate[0.60]*100:.1f}%) | "
        f"val@0.65={mean_dice[0.65]:.4f} (empty {empty_rate[0.65]*100:.1f}%) | "
        f"val@0.70={mean_dice[0.70]:.4f} (empty {empty_rate[0.70]*100:.1f}%) | "
        f"lr_enc={lr_enc:.2e} lr_dec={lr_dec:.2e}"
    )

    if val_score > best_val + 1e-4:
        best_val = val_score
        torch.save(model.state_dict(), best_path)
        no_improve = 0
        print("  ✓ saved best:", best_path, "| best_val@0.65:", best_val)
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print("Early stopping triggered.")
            break

print("Done. Best val@0.65:", best_val)


enc params: 21284672
dec params: 5504929
Start training...
Epoch 001 | train_loss=0.5981 | val@0.60=0.5405 (empty 6.1%) | val@0.65=0.5384 (empty 6.1%) | val@0.70=0.5319 (empty 6.1%) | lr_enc=9.99e-05 lr_dec=9.99e-04
  ✓ saved best: ./resunet_best.pt | best_val@0.65: 0.5384391149562253
Epoch 002 | train_loss=0.4909 | val@0.60=0.5179 (empty 7.1%) | val@0.65=0.4955 (empty 8.1%) | val@0.70=0.4645 (empty 9.1%) | lr_enc=9.96e-05 lr_dec=9.96e-04
Epoch 003 | train_loss=0.4362 | val@0.60=0.3151 (empty 14.1%) | val@0.65=0.2746 (empty 16.2%) | val@0.70=0.2316 (empty 19.2%) | lr_enc=9.91e-05 lr_dec=9.91e-04
Unfreeze encoder (encoder LR tetap kecil via 2LR groups).
Epoch 004 | train_loss=0.3885 | val@0.60=0.5954 (empty 6.1%) | val@0.65=0.5853 (empty 6.1%) | val@0.70=0.5714 (empty 6.1%) | lr_enc=9.84e-05 lr_dec=9.84e-04
  ✓ saved best: ./resunet_best.pt | best_val@0.65: 0.5852989003976222
Epoch 005 | train_loss=0.3365 | val@0.60=0.6638 (empty 3.0%) | val@0.65=0.6626 (empty 3.0%) | val@0.70=0.6590 (e

## Threshold Sweep (Multi-scale)

In [31]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
from tqdm import tqdm

# ===== configs (WAJIB ADA) =====
configs = [
    {"name":"640",         "sizes":(640,),      "use_hflip":False},
    {"name":"640+flip",    "sizes":(640,),      "use_hflip":True},
    {"name":"768",         "sizes":(768,),      "use_hflip":False},
    {"name":"768+flip",    "sizes":(768,),      "use_hflip":True},
    {"name":"640+768",     "sizes":(640,768),   "use_hflip":False},
    {"name":"640+768+flip","sizes":(640,768),   "use_hflip":True},
    # weighted: K probs = 2 sizes * (flip?2:1) => 4 maps
    {"name":"w(0.7@768)+flip", "sizes":(640,768), "use_hflip":True, "weights":[0.15,0.15,0.35,0.35]},  # 640,640flip,768,768flip
    {"name":"w(0.7@640)+flip", "sizes":(640,768), "use_hflip":True, "weights":[0.35,0.35,0.15,0.15]},
]

# ===== cache builder =====
def build_cache_for_config(model, val_imgs, cfg):
    cache = []
    for img_path in tqdm(val_imgs, desc=f"cache[{cfg['name']}]"):
        img_bgr = cv2.imread(img_path)
        if img_bgr is None:
            continue
        mpath = find_mask_for_image(img_path)
        gt = cv2.imread(mpath, cv2.IMREAD_GRAYSCALE)
        if gt is None:
            continue
        g = (gt == 255)

        prob = predict_prob_combo(
            model, img_bgr,
            sizes=cfg["sizes"],
            use_hflip=cfg["use_hflip"],
            weights=cfg.get("weights", None),
        )

        cache.append({"prob": prob.astype(np.float16), "gt": g})
    return cache

# ===== fast sweep from cache =====
def sweep_from_cache(cache, thr_list):
    dices_curve = []
    empties_curve = []
    for thr in thr_list:
        dices = []
        empty = 0
        for item in cache:
            prob = item["prob"]
            gt = item["gt"]
            pred = (prob >= float(thr))
            if pred.sum() == 0:
                empty += 1
            dices.append(dice_coef_np(pred, gt))
        dices_curve.append(float(np.mean(dices)) if dices else 0.0)
        empties_curve.append(float(empty / max(1, len(cache))))
    return np.array(dices_curve, np.float32), np.array(empties_curve, np.float32)

# ===== run comparison =====
thr_list = np.linspace(0.55, 0.85, 31)

plt.figure(figsize=(10,5))
best_summary = []

for cfg in configs:
    cache = build_cache_for_config(model, val_imgs, cfg)   # inference sekali per image
    dice_curve, empty_curve = sweep_from_cache(cache, thr_list)

    best_i = int(np.argmax(dice_curve))
    best_summary.append((cfg["name"], float(thr_list[best_i]), float(dice_curve[best_i]), float(empty_curve[best_i])))
    plt.plot(thr_list, dice_curve, marker=".", label=f'{cfg["name"]} (best {dice_curve[best_i]:.4f})')

plt.xlabel("threshold")
plt.ylabel("val Dice")
plt.title("Dice vs Threshold for multiple TTA configs (cache-based)")
plt.grid(True)
plt.legend()
plt.show()

print("Top configs (name, best_thr, best_dice, empty):")
best_summary = sorted(best_summary, key=lambda x: x[2], reverse=True)
for row in best_summary[:8]:
    print(row)


cache[640]:   0%|          | 0/99 [00:00<?, ?it/s]


NameError: name 'predict_prob_combo' is not defined

<Figure size 1000x500 with 0 Axes>

In [ ]:
# cfg_star sudah dipilih dari best_summary
cache_star = build_cache_for_config(model, val_imgs, cfg_star)

# coarse
thr_list = np.linspace(0.55, 0.85, 31)
dice_curve, empty_curve = sweep_from_cache(cache_star, thr_list)
best_i = int(np.argmax(dice_curve))
best_thr = float(thr_list[best_i])

# refine
ref_lo = max(0.05, best_thr - 0.04)
ref_hi = min(0.95, best_thr + 0.04)
thr_list_ref = np.linspace(ref_lo, ref_hi, 33)

dice_curve_ref, empty_ref = sweep_from_cache(cache_star, thr_list_ref)
best_j = int(np.argmax(dice_curve_ref))
FINAL_THR = float(thr_list_ref[best_j])

import matplotlib.pyplot as plt
plt.figure(figsize=(8,4))
plt.plot(thr_list_ref, dice_curve_ref, marker="o")
plt.axvline(FINAL_THR, linestyle="--")
plt.xlabel("threshold")
plt.ylabel("val Dice")
plt.title(f"Refined sweep — {cfg_star['name']} | best thr={FINAL_THR:.3f}, dice={dice_curve_ref[best_j]:.4f}")
plt.grid(True)
plt.show()

print("FINAL:", cfg_star["name"], "thr=", FINAL_THR, "dice=", float(dice_curve_ref[best_j]), "empty=", float(empty_ref[best_j]))



In [ ]:
def predict_prob_combo_calibrated(model, img_bgr, cfg, gamma=1.0):
    p = predict_prob_combo(model, img_bgr, cfg["sizes"], cfg["use_hflip"], cfg.get("weights", None))
    if gamma != 1.0:
        p = np.clip(p, 1e-6, 1-1e-6) ** float(gamma)
    return p

@torch.no_grad()
def eval_val_dice_tta_config_cal(model, val_img_paths, thr, cfg, gamma=1.0):
    model.eval()
    dices = []
    empty = 0
    total = 0
    for img_path in val_img_paths:
        img_bgr = cv2.imread(img_path)
        if img_bgr is None:
            continue
        mpath = find_mask_for_image(img_path)
        gt = cv2.imread(mpath, cv2.IMREAD_GRAYSCALE)
        if gt is None:
            continue
        g = (gt == 255)

        prob = predict_prob_combo_calibrated(model, img_bgr, cfg, gamma=gamma)
        pred = (prob >= float(thr))
        if pred.sum() == 0:
            empty += 1
        dices.append(dice_coef_np(pred, g))
        total += 1

    return float(np.mean(dices)) if dices else 0.0, float(empty / max(1, total))

gammas = [0.85, 0.90, 0.95, 1.00, 1.05, 1.10]
thr_list = np.linspace(max(0.05, FINAL_THR-0.05), min(0.95, FINAL_THR+0.05), 21)

best_cal = {"gamma":1.0, "thr":FINAL_THR, "dice":-1}
plt.figure(figsize=(9,5))

for gma in gammas:
    dices = []
    for thr in thr_list:
        d, _ = eval_val_dice_tta_config_cal(model, val_imgs, thr, cfg_star, gamma=gma)
        dices.append(d)
    dices = np.array(dices)
    j = int(np.argmax(dices))
    if float(dices[j]) > best_cal["dice"]:
        best_cal = {"gamma":float(gma), "thr":float(thr_list[j]), "dice":float(dices[j])}
    plt.plot(thr_list, dices, marker=".", label=f"gamma={gma} (best {dices[j]:.4f})")

plt.xlabel("threshold")
plt.ylabel("val Dice")
plt.title("Calibration sweep: p^gamma vs threshold")
plt.grid(True)
plt.legend()
plt.show()

print("BEST calibration:", best_cal)


In [ ]:
import matplotlib.pyplot as plt
import random

def show_val_overlays_tta(
    model,
    img_paths,
    cfg,
    thr,
    gamma=1.0,
    n=12,
    seed=42
):
    random.Random(seed).shuffle(img_paths)
    picks = img_paths[:n]

    plt.figure(figsize=(14, 4*n))
    for i, img_path in enumerate(picks, 1):
        img_bgr = cv2.imread(img_path)
        mpath = find_mask_for_image(img_path)
        gt = cv2.imread(mpath, cv2.IMREAD_GRAYSCALE)

        prob = predict_prob_combo_calibrated(model, img_bgr, cfg, gamma=gamma)
        pred = (prob >= float(thr)).astype(np.uint8) * 255

        # FP/FN maps
        gt01 = (gt == 255).astype(np.uint8)
        pr01 = (pred == 255).astype(np.uint8)
        fp = ((pr01 == 1) & (gt01 == 0)).astype(np.uint8)
        fn = ((pr01 == 0) & (gt01 == 1)).astype(np.uint8)

        # show: image, heatmap, pred, fp/fn
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        ax = plt.subplot(n, 4, (i-1)*4+1)
        ax.imshow(img_rgb); ax.set_title(f"Image {os.path.basename(img_path)}"); ax.axis("off")

        ax = plt.subplot(n, 4, (i-1)*4+2)
        ax.imshow(prob, cmap="jet"); ax.set_title("Prob heatmap"); ax.axis("off")

        ax = plt.subplot(n, 4, (i-1)*4+3)
        ax.imshow(gt01, cmap="gray"); 
        ax.imshow(pr01, cmap="Reds", alpha=0.35)
        ax.set_title(f"GT(gray) + Pred(red) thr={thr:.3f}")
        ax.axis("off")

        ax = plt.subplot(n, 4, (i-1)*4+4)
        # FP red, FN blue
        canvas = np.zeros((*gt01.shape, 3), dtype=np.uint8)
        canvas[fp==1] = (255,0,0)
        canvas[fn==1] = (0,0,255)
        ax.imshow(canvas); ax.set_title("FP (red) / FN (blue)"); ax.axis("off")

    plt.tight_layout()
    plt.show()

# show examples
show_val_overlays_tta(model, val_imgs.copy(), cfg_star, thr=best_cal["thr"], gamma=best_cal["gamma"], n=10)


## Inference + Submission (OpenCV)

Kalau `import cv2` gagal:
```
pip install -U opencv-python-headless
```
lalu restart kernel.


In [ ]:
def make_submission_with_cfg(model, test_dir, out_csv, cfg, thr, gamma=1.0):
    import glob, os
    import pandas as pd

    test_files = sorted(glob.glob(os.path.join(str(test_dir), "*")))
    rows = []

    for p in tqdm(test_files, desc="SUBMIT FINAL"):
        img_name = os.path.basename(p)
        img_bgr = cv2.imread(p)
        if img_bgr is None:
            rows.append({"ImageId": img_name, "rle": ""})
            continue

        prob = predict_prob_combo_calibrated(model, img_bgr, cfg, gamma=gamma)
        pred01 = (prob >= float(thr)).astype(np.uint8)
        pred255 = (pred01 * 255).astype(np.uint8)

        rle = encode_rle(pred255, pos_value=255)
        rows.append({"ImageId": img_name, "rle": rle})

    sub = pd.DataFrame(rows)
    sub.to_csv(out_csv, index=False)
    print("Saved:", out_csv, "| cfg:", cfg["name"], "| thr:", thr, "| gamma:", gamma)
    return sub

sub = make_submission_with_cfg(
    model,
    test_dir=str(test_img_dir),
    out_csv="submission_final_ms_tta.csv",
    cfg=cfg_star,
    thr=best_cal["thr"],
    gamma=best_cal["gamma"]
)
sub.head()
